# 실행 순서
1. 문서 조회(document loader - word(docx2txt) 사용)
2. 문서 분할(text spliter)
    - 토큰 수 초과로 답변 생성 불가할 가능성 존재
    - 문서가 길 경우(인풋이 길 경우) 답변 생성 오래걸림
    - Recursively split 사용
        - 이유 : character spliter의 경우 구분자 하나만 사용가능 하지만 Recyrsively는 리스트로 구분자 정의 가능
3. 분할 문서 임베딩(upstage embedding) 후 벡터 데이터베이스(chroma)에 저장
4. 질문이 있을 경우 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 조회한 문서 LLM에 질문과 같이 전달

In [ ]:
%pip install --upgrade --quiet docx2txt langchain-community

In [ ]:
%pip install -qU langchain-text-splitters

# 1. 문서 내용 조회

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 하나의 청크당 토큰 수 
    chunk_overlap=200, # 청크별로 겹치는 수
)

loader = Docx2txtLoader('./data/tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

document_list


In [ ]:
%pip install -U langchain-upstage

In [ ]:
%pip install python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()

# 2. 임베딩

In [ ]:
from langchain_upstage import UpstageEmbeddings

embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
%pip install langchain-chroma

# 3. 벡터 DB 저장

In [ ]:
from langchain_chroma import Chroma

#database = Chroma.from_documents(documents=document_list, embedding=embedding, persist_directory='./chroma_new', collection_name='chroma-tax')
database = Chroma(collection_name='chroma-tax', persist_directory='./chroma_new', embedding_function=embedding)

In [ ]:
print(database._collection.count()) 

# 4. 유사도 검색

In [ ]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요"

#유사도 검색
retrived_docs = database.similarity_search(query, k=3)

retrived_docs

# 5. 모델로 부터 결과 도출

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

In [ ]:
prompt = f"""[Identity]
- 당신은 최고의 한국 소득세 전문가 입니다.
- [Context]를 참고해서 사용자의 질문에 답변해주세요

[Context]
{retrived_docs}

Question:{query}
"""

llm.invoke(prompt).content

In [ ]:
%pip install -U langchain langchainhub langchain-classic

## 5-1. langchain hub에 등록된 프롬프트를 활용해 프롬프트 작성하는 방법

In [ ]:

from langsmith import Client

client = Client()

prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

prompt

# 5-2. langchain hub를 사용하지 않고 프롬프트 작성

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', """소득세법에 대한 질문에 아래 context를 근거로 답변하세요.
질문에 대한 직접적인 문장이 없더라도, 공제 규정과 세율표를 활용해 단계적으로 계산할 수 있다면 계산 과정을 보여주고 답하세요.
context에서 전혀 근거를 찾을 수 없을 때만 모른다고 답하세요.

    [Context]
    {context}
    """),
    ('human', '{question}')
])


# 6. QA chain 활용하여 llm 결과 도출

In [ ]:
# QA chain

from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

ai_message = qa_chain.invoke({"query": query})

In [73]:
ai_message['result']

'연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 몇 가지 단계를 거쳐야 합니다. 이때, 제공된 context와 일반적인 세법 지식을 활용하여 계산하겠습니다.\n\n1. **근로소득공제**: 연봉 5천만원에 대해 근로소득공제를 계산해야 합니다. 근로소득공제의 규정에 따라, 공제가 2천만원을 초과하는 경우에는 2천만원을 공제합니다.\n\n2. **종합소득과세표준**: 종합소득과세표준은 총급여액에서 근로소득공제를 차감하여 계산합니다.\n   \\[\n   종합소득과세표준 = 5천만원 - 2천만원 (근로소득공제) = 3천만원\n   \\]\n\n3. **종합소득세액**: 제공된 context에서 종합소득에 대한 구체적인 세율표가 제공되지 않았으므로 보통의 세율을 적용합니다. 2023년 기준으로 기본적인 소득세율은 다음과 같으며, 이를 계산에 활용합니다:\n\n   - 과세표준 1,200만원 이하: 6%\n   - 과세표준 1,200만원 ~ 4,600만원: 15%\n   - 과세표준 4,600만원 ~ 8,800만원: 24%\n   - 그 이후 세율도 존재하지만 이 경우엔 해당되지 않습니다.\n\n   따라서, 해당과세표준의 세금을 계산하도록 하겠습니다.\n   \\[\n   소득세 = 1,200만원 \\times 0.06 + (3,000만원 - 1,200만원) \\times 0.15\n   \\]\n   \\[\n   소득세 = 72만원 + (1,800만원 \\times 0.15) = 72만원 + 270만원 = 342만원\n   \\]\n\n4. **근로소득세액공제 및 기타 공제**: 종합소득산출세액에서 근로소득세액공제를 적용합니다. 총급여액 5천만원이므로, 근로소득세액공제는 74만원 - [(5천만원 - 3,300만원) × 8/1000]입니다. 계산하면:\n\n   \\[\n   공제액 = 74만원 - [(5천만원 - 3,300만원) \\times 0.008] = 74만원 - 13.6만원 = 60.4만원\n   \\]\n\n   따라서, 최종 소득세는:\n   \